# Structural Hypertransmission Pipeline Validation

This notebook applies the structural-hypertransmission detector to one or more
selected OCT B-scans.

For every selected scan, the same processing sequence is applied:

1. Load the Heidelberg E2E volume.
2. Extract the requested B-scan.
3. Flatten the scan to Bruch's membrane (BM).
4. Crop a fixed 150-pixel region below BM.
5. Apply whole-ROI z-score intensity normalization.
6. Apply anisotropic Gaussian denoising.
7. Construct a gradient-gated structural exclusion mask.
8. Estimate structurally cleaned column-level intensity.
9. Identify conservative hypertransmission candidates.
10. Refine candidates using depth continuity.
11. Refine persistent candidates using vertical organization.
12. Extract contiguous candidate barcoding intervals.

## Preprocessing configuration

The preprocessing configuration is fixed across all scans.

### Flattening

The BM annotation is retrieved for the selected B-scan and missing coordinates
are interpolated when necessary. Each image column is vertically shifted so that
BM lies on a common horizontal reference row.

### Cropping

A 150-pixel region beginning at BM and extending below BM is retained for
analysis.

### Intensity normalization

Whole-ROI z-score normalization is applied:

$$
I_{\mathrm{norm}}(z,x)
=
\frac{
I(z,x)-\mu
}{
\sigma
},
$$

where:

- $I(z,x)$ is the pixel intensity at depth \(z\) and horizontal position \(x\);
- $\mu$ is the mean intensity of the complete cropped ROI;
- $\sigma$ is the standard deviation of the complete cropped ROI.

### Denoising

An anisotropic Gaussian filter is applied using:

- depth sigma = 1.0 pixels;
- horizontal sigma = 0.5 pixels.

The larger depth-wise smoothing reduces speckle while limiting horizontal
blurring of narrow vertical structures.

## Detector configuration

The detector configuration is also held fixed across scans.

Structural evidence is first defined using simple gradient-based verticality
combined with an 80th-percentile gradient-magnitude requirement.

Column median and column 90th-percentile intensity are then calculated after
structural exclusion. Robust intensity thresholds are defined from the
scan-level column distributions.

Candidate hypertransmission is subsequently refined using:

- the 60th percentile of local depth continuity;
- the 70th percentile of strong vertical-pixel fraction.

Short positive detections and small internal gaps are cleaned before contiguous
candidate intervals are returned.

Quantile-based numerical thresholds are recalculated within each scan, but the
quantile rules themselves remain fixed.

In [1]:
from __future__ import annotations
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "src").exists():
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise RuntimeError("Could not locate the project root.")
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0,str(PROJECT_ROOT))

from src.detector.data import build_grouped_volume_registry, load_e2e_volume
from src.detector.preprocessing import preprocess_bscan
from src.detector.detector import detect_structural_hypertransmission

print( "Project root:",PROJECT_ROOT)

Project root: C:\Users\u0667281\Desktop\JMA\barcoding-amd


## Cohort and Scan Selection

`SCAN_CONFIG` controls which scans are processed.
Any number of subjects may be supplied, and each subject may contain any number
of B-scan indices.

In [2]:
E2E_DIRECTORY = (PROJECT_ROOT / "data" / "heyex" / "meta")
PROGRESSION_GROUPS = {"fast": [8,9,12,41,49],
                      "slow": [17,23,35,36,47]}

registry = build_grouped_volume_registry(e2e_directory=E2E_DIRECTORY,progression_groups=PROGRESSION_GROUPS)
print("Registry entries:", len(registry))

Registry entries: 10


In [3]:
SCAN_CONFIG = {8: list(range(47,56))}

# SCAN_CONFIG = {
#     8: [48],
#     9: [85],
#     12: [38],
#     41: [59],
#     49: [33],
#     17: [55],
#     23: [43],
#     35: [38],
#     36: [59],
#     47: [31],
# }

In [4]:
PREPROCESSING_CONFIG = {
    # Anatomical reference
    "layer_name": "BM",

    # Flattening
    "reference_row": None,
    "flatten_fill_value": 0.0,

    # Sub-BM crop
    "depth_below_layer": 150,
    "include_boundary": True,
    "require_full_depth": False,
    "crop_fill_value": 0.0,

    # Intensity normalization
    "normalization_method": "zscore",
    "lower_percentile": 1.0,
    "upper_percentile": 99.0,

    # Gaussian denoising
    "denoise_method": "gaussian",
    "gaussian_sigma": (
        1.0, # z
        0.5, # x
    ),
}

In [5]:
DETECTOR_CONFIG = {
    # Pixel-level verticality
    "verticality_smoothing_sigma": 1.0,
    "verticality_threshold": 0.60,

    # Gradient-gated structural exclusion
    "gradient_quantile": 0.80,
    "minimum_component_size": 0,

    # Structurally cleaned column intensity
    "column_upper_quantile": 0.90,
    "minimum_valid_pixels": 5,

    # One-dimensional signal smoothing
    "signal_smoothing_sigma": 2.0,

    # Robust intensity thresholds
    "median_iqr_multiplier": 1.0,
    "q90_iqr_multiplier": 0.5,

    # Depth continuity
    "continuity_window_width": 15,
    "continuity_depth_lag": 4,
    "continuity_minimum_row_standard_deviation": 1e-6,
    "continuity_quantile": 0.60,

    # Final vertical refinement
    "vertical_fraction_quantile": 0.70,

    # Spatial cleanup
    "minimum_positive_run": 5,
    "maximum_negative_gap": 2,

    # Edge handling
    "edge_margin": 10,
}

In [6]:
def resolve_subject_record(
    registry,
    subject_id: int,
):
    """
    Resolve one registry record for a requested subject.
    """
    matches = [record for record in registry if int(record.subject_id) == int(subject_id)]

    if not matches:
        raise KeyError(f"Subject {subject_id} was not found in the volume registry.")
    if len(matches) > 1:
        raise ValueError(
            f"Subject {subject_id} has {len(matches)} volume records. "
            "Use explicit volume selection before running this notebook."
        )
    return matches[0]

In [7]:
def plot_pipeline_result(
    *,
    subject_id: int,
    progression_group: str,
    bscan_index: int,
    processed,
    detector_result,
    overlay_color: str = "tab:red",
    overlay_alpha: float = 0.30,
    figure_size: tuple[float, float] = (
        14,
        18,
    ),
):
    """
    Display all major preprocessing stages followed by the detector result.
    """
    stage_images = (
        ("Raw B-scan",processed.raw_bscan,),
        ("Flattened to BM",processed.flattened_bscan,),
        ("150-pixel sub-BM crop",processed.sub_layer_crop,),
        ("Whole-ROI z-score normalization",processed.normalized_scan,),
        ("Gaussian-denoised scan",processed.denoised_scan,),
    )

    figure, axes = plt.subplots(nrows=6,ncols=1,figsize=figure_size,)

    for axis, (title,image,) in zip(axes[:5],stage_images,):
        axis.imshow(image,cmap="gray",aspect="auto",)
        axis.set_title(title)
        axis.set_ylabel("Axial position")
        axis.set_xlabel("Horizontal position")

    detector_image = np.asarray(processed.denoised_scan,dtype=np.float32,)
    image_height, image_width = (detector_image.shape)
    axes[5].imshow(detector_image,cmap="gray",aspect="auto",extent=(
            -0.5,image_width - 0.5,image_height - 0.5,-0.5,
        ),
    )

    for interval in detector_result.intervals:
        axes[5].axvspan(interval.start,interval.end,color=overlay_color,alpha=overlay_alpha,linewidth=0,)
    axes[5].set_title("Final candidate barcoding intervals")
    axes[5].set_ylabel("Depth below BM")
    axes[5].set_xlabel("Horizontal position")
    axes[5].set_xlim(-0.5,image_width - 0.5,)
    axes[5].set_ylim(image_height - 0.5, -0.5,)

    figure.suptitle(
        f"Subject {subject_id} | "
        f"{progression_group.title()} progressor | "
        f"B-scan {bscan_index}",
        fontsize=15,
        y=0.995,
    )

    figure.tight_layout(rect=(0,0,1,0.985,))

    return figure, axes

## Run Pipeline

Each requested E2E volume is loaded once.

Every requested B-scan is then:

- preprocessed using the fixed preprocessing configuration;
- passed to the structural-hypertransmission detector;
- displayed as an independent six-stage figure;
- summarized into a scan-level result table.

No parameters are modified between scans.

In [8]:
pipeline_results = {}
summary_rows = []

for subject_id, scan_indices in (SCAN_CONFIG.items()):
    record = resolve_subject_record(registry,subject_id=subject_id)
    volume = load_e2e_volume(record.e2e_path)
    progression_group = str(record.progression_group).lower()

    print(f"\nSubject {subject_id}")
    print("Progression group:", progression_group)
    print("E2E volume:",record.e2e_path)
    print("Number of B-scans:",len(volume))

    for bscan_index in scan_indices:
        if not (0 <= int(bscan_index) < len(volume)):
            raise IndexError( f"Subject {subject_id}: "
                f"B-scan {bscan_index} is outside "
                f"the valid range 0 to "
                f"{len(volume) - 1}.")

        processed = preprocess_bscan( volume=volume,bscan_index=int(bscan_index),**PREPROCESSING_CONFIG)
        detector_result = (detect_structural_hypertransmission(processed.denoised_scan,config=DETECTOR_CONFIG))
        result_key = (int(subject_id),int(bscan_index))
        pipeline_results[result_key] = {
            "record": record,
            "processed": processed,
            "detector": detector_result,
        }

        interval_list = [(int(interval.start),int(interval.end),) for interval in detector_result.intervals]

        summary_rows.append({
                "subject_id":int( subject_id),
                "progression_group": (progression_group),
                "bscan_index": int(bscan_index),
                "number_of_intervals": int(len(detector_result.intervals)),
                "intervals": (interval_list),
            }
        )

        plot_pipeline_result(subject_id=int(subject_id),
                             progression_group=(progression_group),
                             bscan_index=int(bscan_index),
                             processed=processed,
                             detector_result=(detector_result),
                             overlay_color=("tab:red" if progression_group == "fast" else "tab:green"))
        plt.show()

        print(
            f"Completed B-scan "
            f"{bscan_index}: "
            f"{len(detector_result.intervals)} "
            "candidate interval(s)"
        )


Subject 8
Progression group: fast
E2E volume: C:\Users\u0667281\Desktop\JMA\barcoding-amd\data\heyex\meta\ea8.E2E
Number of B-scans: 97


NameError: name 'gaussian_filter' is not defined

## Scan-Level Detection Summary

The table below contains one row per processed B-scan.

For each scan it reports:

- subject identifier;
- progression group;
- B-scan index;
- number of detected candidate barcoding intervals;
- horizontal coordinates of each detected interval.

Intervals are represented as inclusive `(start, end)` image-column coordinates.

In [ ]:
detection_summary = pd.DataFrame(summary_rows)
detection_summary = (detection_summary
                     .sort_values(["subject_id","bscan_index"])
                     .reset_index(drop=True))

detection_summary